In [1]:
#Cell 1 — load everything (including baseline results):
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import joblib

from src.utils import load_config, set_seed
from src.models.advanced import train_random_forest, train_xgboost
from src.evaluation.metrics import evaluate_model, print_metrics, results_to_dataframe

cfg = load_config()
set_seed(cfg["project"]["random_state"])
processed_dir = cfg["paths"]["processed_dir"]

X_train_raw = pd.read_parquet(os.path.join(processed_dir, "X_train_raw.parquet"))
X_test_raw = pd.read_parquet(os.path.join(processed_dir, "X_test_raw.parquet"))
y_train = pd.read_parquet(os.path.join(processed_dir, "y_train.parquet"))["label_binary"]
y_test = pd.read_parquet(os.path.join(processed_dir, "y_test.parquet"))["label_binary"]

# Load previous baseline results so we can build one combined table
baseline_results = pd.read_csv(os.path.join(cfg["paths"]["metrics_dir"], "baseline_results.csv"))
print("Baseline results loaded:")
print(baseline_results[["model", "f1", "roc_auc"]])

Baseline results loaded:
                 model        f1   roc_auc
0        Decision Tree  0.999629  0.999844
1  Logistic Regression  0.998946  0.999888


In [2]:
#Cell 2 — train + evaluate Random Forest:
seed = cfg["project"]["random_state"]

rf_model, rf_train_time = train_random_forest(X_train_raw, y_train, random_state=seed)
rf_metrics = evaluate_model(rf_model, X_test_raw, y_test, model_name="Random Forest")
rf_metrics["train_time_sec"] = round(rf_train_time, 4)

print_metrics(rf_metrics)

RESULTS: Random Forest
Accuracy   : 0.9999
Precision  : 1.0000
Recall     : 0.9999
F1-score   : 0.9999
ROC-AUC    : 1.0000
PR-AUC     : 1.0000
FPR        : 0.0000
FNR        : 0.0001
------------------------------------------------------------
Confusion Matrix:
              Predicted BENIGN   Predicted DDoS
Actual BENIGN        19,013                 0
Actual DDoS               3            25,600
------------------------------------------------------------
Training time    : 21.6514 sec
Prediction time  : 0.2982 sec


In [3]:
#Cell 3 — train + evaluate XGBoost:
xgb_model, xgb_train_time = train_xgboost(X_train_raw, y_train, random_state=seed)
xgb_metrics = evaluate_model(xgb_model, X_test_raw, y_test, model_name="XGBoost")
xgb_metrics["train_time_sec"] = round(xgb_train_time, 4)

print_metrics(xgb_metrics)

RESULTS: XGBoost
Accuracy   : 1.0000
Precision  : 1.0000
Recall     : 1.0000
F1-score   : 1.0000
ROC-AUC    : 1.0000
PR-AUC     : 1.0000
FPR        : 0.0000
FNR        : 0.0000
------------------------------------------------------------
Confusion Matrix:
              Predicted BENIGN   Predicted DDoS
Actual BENIGN        19,013                 0
Actual DDoS               1            25,602
------------------------------------------------------------
Training time    : 3.5311 sec
Prediction time  : 0.1076 sec


In [4]:
#Cell 4 — combined comparison table (all 4 models) + save:
all_results = results_to_dataframe(
    baseline_results.to_dict("records") + [rf_metrics, xgb_metrics]
)
print(all_results)

metrics_dir = cfg["paths"]["metrics_dir"]
models_dir = cfg["paths"]["models_dir"]

all_results.to_csv(os.path.join(metrics_dir, "all_models_comparison.csv"), index=False)

joblib.dump(rf_model, os.path.join(models_dir, "random_forest.joblib"))
joblib.dump(xgb_model, os.path.join(models_dir, "xgboost.joblib"))

print("\nSaved combined comparison table and both models.")

                 model  accuracy  precision    recall        f1   roc_auc  \
0              XGBoost  0.999978   1.000000  0.999961  0.999980  1.000000   
1        Random Forest  0.999933   1.000000  0.999883  0.999941  1.000000   
2        Decision Tree  0.999574   0.999805  0.999453  0.999629  0.999844   
3  Logistic Regression  0.998790   0.998867  0.999024  0.998946  0.999888   

     pr_auc       fpr       fnr  train_time_sec  prediction_time_sec     tn  \
0  1.000000  0.000000  0.000039          3.5311               0.1076  19013   
1  1.000000  0.000000  0.000117         21.6514               0.2982  19013   
2  0.999782  0.000263  0.000547          2.4128               0.0483  19008   
3  0.999803  0.001525  0.000976          4.9296               0.0294  18984   

   fp  fn     tp  
0   0   1  25602  
1   0   3  25600  
2   5  14  25589  
3  29  25  25578  

Saved combined comparison table and both models.


In [5]:
#Cell 5 — feature importance check (same due-diligence as Step 4):
import pandas as pd

rf_importances = pd.Series(rf_model.feature_importances_, index=X_train_raw.columns).sort_values(ascending=False)
xgb_importances = pd.Series(xgb_model.feature_importances_, index=X_train_raw.columns).sort_values(ascending=False)

print("Random Forest - top 10 features:")
print(rf_importances.head(10))
print(f"\nTop 3 features (RF) account for {rf_importances.iloc[:3].sum()*100:.2f}% of importance")

print("\n" + "="*50)
print("XGBoost - top 10 features:")
print(xgb_importances.head(10))
print(f"\nTop 3 features (XGB) account for {xgb_importances.iloc[:3].sum()*100:.2f}% of importance")

Random Forest - top 10 features:
Fwd Packet Length Max     0.107586
Avg Fwd Segment Size      0.069147
Fwd Packet Length Mean    0.062918
act_data_pkt_fwd          0.062815
Init_Win_bytes_forward    0.048855
Subflow Fwd Bytes         0.046785
Avg Bwd Segment Size      0.046050
Bwd Packet Length Std     0.045687
Destination Port          0.041417
Subflow Fwd Packets       0.039571
dtype: float64

Top 3 features (RF) account for 23.97% of importance

XGBoost - top 10 features:
Fwd Packet Length Max          0.669322
Total Length of Fwd Packets    0.310571
Fwd Packet Length Mean         0.008705
Bwd Packet Length Mean         0.002213
Init_Win_bytes_forward         0.001225
Bwd Header Length              0.000835
Active Mean                    0.000718
FIN Flag Count                 0.000702
Packet Length Std              0.000562
Destination Port               0.000543
dtype: float32

Top 3 features (XGB) account for 98.86% of importance
